In [19]:
import keras_tuner as kt
import tensorflow as tf
from tensorflow.keras import layers,models


In [20]:
import tensorflow as tf

# 1. Update this path to match Kaggle's filesystem
DATASET_PATH = "/kaggle/input/datasets/emmarex/plantdisease/PlantVillage"
IMG_HEIGHT, IMG_WIDTH = 224, 224
BATCH_SIZE = 32
SEED = 42
EPOCHS = 10

# 2. Create the Training dataset (70%)
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.3,
    subset="training",
    seed=SEED,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE
)

# 3. Create the temporary dataset for splitting (30%)
temp_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.3,
    subset="validation",
    seed=SEED,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE
)

# 4. Split the 30% into 15% Validation and 15% Test
temp_batches = tf.data.experimental.cardinality(temp_ds)
val_ds = temp_ds.take(temp_batches // 2)
test_ds = temp_ds.skip(temp_batches // 2)

NUM_CLASSES = 15

# 5. Optimize performance with Autotune Prefetching
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

# 6. Your pre-calculated class weights for handling data imbalance
class_weights = {
    0: 1.3858033573141486, 1: 0.9451750081779522, 2: 1.347039627039627,
    3: 1.347039627039627, 4: 9.631333333333334, 5: 0.6534147444595205,
    6: 1.358439116125999, 7: 0.7050756466569058, 8: 1.4311045071817732,
    9: 0.7907498631636563, 10: 0.8288582903040734, 11: 0.9451750081779522,
    12: 0.4278690952169406, 13: 3.821957671957672, 14: 0.8926166203274637
}

print("\nData pipeline successfully initialized on Kaggle!")

Found 20638 files belonging to 15 classes.
Using 14447 files for training.
Found 20638 files belonging to 15 classes.
Using 6191 files for validation.

Data pipeline successfully initialized on Kaggle!


In [21]:
def build_model(hp):
    model = models.Sequential()
    model.add(layers.Input(shape=(224,224,3)))

    model.add(layers.Rescaling(1./255))
    model.add(layers.RandomFlip("horizontal"))

    model.add(layers.RandomRotation(hp.Float("rotation",min_value=0.05,max_value=0.2,step=0.05)))

    # ------------------
    # BLOCK 1
    # ------------------

    model.add(layers.Conv2D(filters=hp.Choice("conv1_filters",[32,64]),kernel_size=(3,3),activation="relu",padding="same"))

    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D())

    # ------------------
    # BLOCK 2
    # ------------------

    model.add(layers.Conv2D(filters=hp.Choice("conv2_filters",[64,128]),kernel_size=(3,3),activation="relu",padding="same"))

    model.add(layers.BatchNormalization())

    model.add(layers.MaxPooling2D())

    # ------------------
    # BLOCK 3
    # ------------------

    model.add(layers.Conv2D(filters=hp.Choice("conv3_filters",[128,256]),kernel_size=(3,3),activation="relu",padding="same"))

    model.add(layers.BatchNormalization())

    model.add(layers.MaxPooling2D())

    # ------------------
    # BLOCK 4
    # ------------------

    model.add(

        layers.Conv2D(filters=hp.Choice("conv4_filters",[256,512]),kernel_size=(3,3),activation="relu",padding="same"))

    model.add(layers.BatchNormalization())

    model.add(layers.MaxPooling2D())

    model.add(layers.GlobalAveragePooling2D())

    model.add(layers.Dense(units=hp.Choice("dense_units",[128,256,512]),activation="relu"))

    model.add(layers.Dropout(hp.Float("dropout",min_value=0.2,max_value=0.6,step=0.1)))

    model.add(layers.Dense(NUM_CLASSES,activation="softmax"))

    optimizer_choice = hp.Choice("optimizer",["adam","rmsprop"])

    learning_rate = hp.Choice("learning_rate",[1e-3,1e-4,5e-4])

    if optimizer_choice == "adam":
        optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)

    else:
        optimizer = tf.keras.optimizers.RMSprop(learning_rate=learning_rate)

    model.compile(optimizer=optimizer,loss="sparse_categorical_crossentropy",metrics=["accuracy"])
    return model

In [25]:
tuner = kt.RandomSearch(
    build_model,
    objective="val_accuracy",
    max_trials=2,
    executions_per_trial=1,
    directory="keras_tuner",
    project_name="plant_disease")

tuner.search_space_summary()

Reloading Tuner from keras_tuner/plant_disease/tuner0.json
Search space summary
Default search space size: 9
rotation (Float)
{'default': 0.05, 'conditions': [], 'min_value': 0.05, 'max_value': 0.2, 'step': 0.05, 'sampling': 'linear'}
conv1_filters (Choice)
{'default': 32, 'conditions': [], 'values': [32, 64], 'ordered': True}
conv2_filters (Choice)
{'default': 64, 'conditions': [], 'values': [64, 128], 'ordered': True}
conv3_filters (Choice)
{'default': 128, 'conditions': [], 'values': [128, 256], 'ordered': True}
conv4_filters (Choice)
{'default': 256, 'conditions': [], 'values': [256, 512], 'ordered': True}
dense_units (Choice)
{'default': 128, 'conditions': [], 'values': [128, 256, 512], 'ordered': True}
dropout (Float)
{'default': 0.2, 'conditions': [], 'min_value': 0.2, 'max_value': 0.6, 'step': 0.1, 'sampling': 'linear'}
optimizer (Choice)
{'default': 'adam', 'conditions': [], 'values': ['adam', 'rmsprop'], 'ordered': False}
learning_rate (Choice)
{'default': 0.001, 'conditions'

In [23]:
early_stop = tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=4,restore_best_weights=True)

tuner.search(train_ds,validation_data=val_ds,epochs=15,class_weight=class_weights,callbacks=[early_stop])

best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]
print(best_hp.values)

Trial 2 Complete [00h 14m 38s]
val_accuracy: 0.8949742317199707

Best val_accuracy So Far: 0.8949742317199707
Total elapsed time: 00h 24m 45s
{'rotation': 0.05, 'conv1_filters': 64, 'conv2_filters': 64, 'conv3_filters': 128, 'conv4_filters': 512, 'dense_units': 512, 'dropout': 0.2, 'optimizer': 'rmsprop', 'learning_rate': 0.001}


In [24]:
best_model = tuner.get_best_models(num_models=1)[0]

test_loss, test_acc = best_model.evaluate(test_ds)
print("Test Accuracy:", test_acc)
best_model.save("best_tuned_cnn.keras")

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 22 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


97/97 ━━━━━━━━━━━━━━━━━━━━ 6s 37ms/step - accuracy: 0.8772 - loss: 0.4178
Test Accuracy: 0.8772270679473877
